# Fine-tuning Médical — TechCorp IA Hackathon

Fine-tuning QLoRA de `microsoft/Phi-3.5-mini-instruct` sur le dataset médical `ruslanmv/ai-medical-chatbot`.

**Objectif :** Spécialiser le modèle pour répondre à des questions médicales conversationnelles.

**Infrastructure :** Google Colab Pro (GPU T4/A100) — modèle expérimental, pas pour production.

---
## Étape 1 — Installation des dépendances

In [ ]:
# Installation des dépendances pour le fine-tuning QLoRA
!pip install -q transformers==4.44.0 peft==0.12.0 bitsandbytes==0.43.3 \
    trl==0.11.1 datasets==3.0.1 accelerate==0.34.2
print("✅ Dépendances installées")

## Étape 2 — Imports et configuration

Détection automatique du GPU disponible et configuration du dtype optimal.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32
BASE_MODEL = "microsoft/Phi-3.5-mini-instruct"
OUTPUT_DIR = "/content/medical_model"

print(f"Device  : {DEVICE}")
print(f"dtype   : {DTYPE}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Étape 3 — Chargement du dataset médical

Utilisation du dataset `ruslanmv/ai-medical-chatbot` (HuggingFace).  
Colonnes : `Patient` (question) et `Doctor` (réponse).

In [ ]:
DATASET_LIMIT = 5000

print(f"Chargement ruslanmv/ai-medical-chatbot (limite: {DATASET_LIMIT} entrées)...")
raw_dataset = load_dataset("ruslanmv/ai-medical-chatbot", split="train")
raw_dataset = raw_dataset.select(range(min(DATASET_LIMIT, len(raw_dataset))))

print(f"✅ {len(raw_dataset)} entrées chargées")
print(f"Colonnes : {raw_dataset.column_names}")
print("\nExemple :")
print(f"  Patient : {raw_dataset[0].get('Patient', '')[:120]}")
print(f"  Doctor  : {raw_dataset[0].get('Doctor', '')[:120]}")

## Étape 4 — Formatage du dataset au format ChatML (Phi-3.5)

Phi-3.5 utilise le template : `<|user|>\n{message}<|end|>\n<|assistant|>\n{response}<|end|>`

In [ ]:
def format_chatml(example):
    question = (example.get("Patient") or "").strip()
    answer = (example.get("Doctor") or "").strip()
    if not question or not answer:
        return {"text": None}
    text = f"<|user|>\n{question}<|end|>\n<|assistant|>\n{answer}<|end|>"
    return {"text": text}

formatted = raw_dataset.map(format_chatml, remove_columns=raw_dataset.column_names)
formatted = formatted.filter(lambda x: x["text"] is not None)

print(f"✅ {len(formatted)} exemples formatés")
print("\nExemple formaté :")
print(formatted[0]["text"][:300])

## Étape 5 — Chargement du modèle de base avec QLoRA 4-bit

QLoRA (Quantized LoRA) permet d'entraîner un modèle 3.8B en 4-bit sur ~8GB de VRAM.  
La quantification NF4 préserve les performances tout en réduisant la mémoire de ~75%.

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f"Chargement {BASE_MODEL} en 4-bit...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16,
)
model = prepare_model_for_kbit_training(model)
print(f"✅ Modèle chargé — Paramètres totaux: {model.num_parameters():,}")

## Étape 6 — Configuration LoRA

LoRA (Low-Rank Adaptation) ajoute des matrices d'adaptation de rang faible (r=8) sur les couches d'attention.  
Seuls ~0.5% des paramètres sont entraînés, ce qui accélère l'entraînement de ~10x.

In [ ]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["qkv_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
trainable, total = model.get_nb_trainable_parameters()
print(f"✅ LoRA appliqué")
print(f"   Paramètres entraînables : {trainable:,} ({100*trainable/total:.2f}%)")
print(f"   Paramètres totaux       : {total:,}")

## Étape 7 — Configuration de l'entraînement (SFTTrainer)

SFT (Supervised Fine-Tuning) avec `trl.SFTTrainer`.  
- 2 epochs pour rester dans les limites de temps Colab
- Gradient accumulation x4 pour simuler un batch effectif de 8
- `max_seq_length=512` adapté aux conversations médicales

In [ ]:
sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_steps=50,
    logging_steps=25,
    save_steps=200,
    save_total_limit=2,
    fp16=True,
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    max_seq_length=512,
    dataset_text_field="text",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=formatted,
    processing_class=tokenizer,
)

steps_per_epoch = len(formatted) // (sft_config.per_device_train_batch_size * sft_config.gradient_accumulation_steps)
print(f"✅ Trainer configuré")
print(f"   Exemples         : {len(formatted)}")
print(f"   Steps par epoch  : {steps_per_epoch}")
print(f"   Steps totaux     : {steps_per_epoch * sft_config.num_train_epochs}")

## Étape 8 — Entraînement

La loss doit descendre progressivement. Une loss finale < 1.5 indique un bon apprentissage.  
Durée estimée : ~30-45 min sur T4, ~15-20 min sur A100.

In [ ]:
import time

print("Démarrage de l'entraînement...\n")
t0 = time.time()

train_result = trainer.train()

elapsed = time.time() - t0
print(f"\n✅ Entraînement terminé en {elapsed/60:.1f} minutes")
print(f"   Loss finale      : {train_result.training_loss:.4f}")
print(f"   Steps effectués  : {train_result.global_step}")
print(f"   Loss par step    :")
for log in trainer.state.log_history:
    if "loss" in log:
        print(f"     step={log.get('step', '?'):4d}  epoch={log.get('epoch', '?'):.2f}  loss={log['loss']:.4f}")

## Étape 9 — Sauvegarde du modèle

Sauvegarde de l'adapter LoRA et du tokenizer dans `/content/medical_model/`.  
Ce répertoire peut être téléchargé ou monté sur Google Drive.

In [ ]:
import os

os.makedirs(OUTPUT_DIR, exist_ok=True)
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

files = os.listdir(OUTPUT_DIR)
print(f"✅ Modèle sauvegardé dans {OUTPUT_DIR}")
print(f"   Fichiers : {files}")

# Optionnel : monter sur Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r /content/medical_model /content/drive/MyDrive/medical_model

## Étape 10 — Test du modèle fine-tuné

Validation qualitative avec 3 questions médicales représentatives.  
Les réponses doivent être précises, contextualisées et sans hallucinations grossières.

In [ ]:
from peft import PeftModel

# Recharger le modèle fine-tuné pour les tests
print("Chargement du modèle fine-tuné pour test...")
test_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16,
)
test_model = PeftModel.from_pretrained(test_model, OUTPUT_DIR)
test_model.eval()

MEDICAL_QUESTIONS = [
    "I have a persistent headache for 3 days with sensitivity to light. What could it be?",
    "What are the main differences between type 1 and type 2 diabetes?",
    "My child has a fever of 38.5°C and a sore throat. Should I be worried?",
]

def generate(question, max_new_tokens=200):
    prompt = f"<|user|>\n{question}<|end|>\n<|assistant|>\n"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = test_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    input_len = inputs["input_ids"].shape[1]
    response = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)
    return response.strip()

print("\n" + "="*60)
for i, q in enumerate(MEDICAL_QUESTIONS, 1):
    print(f"\n[{i}] Patient: {q}")
    answer = generate(q)
    print(f"    Doctor : {answer[:400]}")
    print("-"*60)

print("\n✅ Tests terminés — modèle expérimental validé (non destiné à la production)")